# Seeing Predictor — Exploratory Data Analysis (real ERA5 data)

Analysis of the XGBoost atmospheric-seeing model trained on **3 years of ECMWF
ERA5 reanalysis** for Stanford, CA (2023–2025). The seeing label is derived
from the optical-turbulence profile of the reanalysis pressure levels
(Cn² integral → Fried parameter → FWHM; see `api/ml/era5.py`).

The model is a multi-quantile booster predicting P10 / P50 / P90 of seeing.
This notebook inspects the label distribution, which weather variables drive
the prediction, calibration on a chronological holdout, and a sample forecast
sequence.

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb

# Put the repo root on sys.path so `api.ml.*` imports resolve when the
# notebook runs from api/ml/notebooks/.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from api.ml.features import FEATURE_NAMES, N_FEATURES
from api.ml.train_xgb import QUANTILES

DATA_PATH = os.path.join(REPO_ROOT, "api", "ml", "data", "stanford_3yr.npz")
MODEL_PATH = os.path.join(REPO_ROOT, "api", "ml", "models", "seeing_model.json")

data = np.load(DATA_PATH)
X, y = data["X"], data["y"]
print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"features: {N_FEATURES}")
print(f"seeing  : min {y.min():.3f}  median {np.median(y):.3f}  max {y.max():.3f} arcsec")
print(f"quantiles predicted: {QUANTILES}")

# Chronological 80/20 holdout, matching api/ml/train_xgb.py (no shuffle).
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print(f"train {len(X_train)}   val {len(X_val)}")

## 2. Seeing distribution

Histogram of the derived seeing labels with a maximum-likelihood log-normal
fit overlaid. Atmospheric seeing is classically log-normal, so a clean fit is a
sanity check on the Cn² → FWHM label-derivation chain.

In [ ]:
# Log-normal MLE: fit a normal to log(seeing).
logs = np.log(y)
mu, sigma = float(logs.mean()), float(logs.std())
p10, p50, p90 = np.percentile(y, [10, 50, 90])
print(f"median {p50:.3f}   P10 {p10:.3f}   P90 {p90:.3f} arcsec")
print(f"log-normal fit: mu={mu:.3f}  sigma={sigma:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(y, bins=40, density=True, alpha=0.6, color="steelblue", label="ERA5 seeing")
xs = np.linspace(y.min(), y.max(), 300)
pdf = np.exp(-((np.log(xs) - mu) ** 2) / (2 * sigma ** 2)) / (xs * sigma * np.sqrt(2 * np.pi))
ax.plot(xs, pdf, "r-", lw=2, label="log-normal fit")
ax.axvline(p50, color="k", ls="--", lw=1, label=f"median {p50:.2f}\"")
ax.set_xlabel("seeing FWHM (arcsec)")
ax.set_ylabel("density")
ax.set_title("Stanford ERA5 seeing distribution (2023–2025)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Feature importance

The key scientific result: which of the 27 weather features the trees actually
split on (by gain). The pressure-level wind-shear features
(`wind_shear_850_300`, `wind_shear_500_200`) and `tropopause_stability`
encode the free-atmosphere turbulence that drives optical seeing, so they are
expected to rank highly on real data.

In [ ]:
booster = xgb.Booster()
booster.load_model(MODEL_PATH)
print("model features:", booster.num_features())

fig, ax = plt.subplots(figsize=(8, 9))
xgb.plot_importance(booster, ax=ax, importance_type="gain", show_values=False)
ax.set_title("Feature importance (gain) — 27 features")
plt.tight_layout()
plt.show()

## 4. Predicted vs actual (calibration)

P50 prediction vs actual seeing on the chronological validation holdout, with
the P10–P90 interval shown as error bars and a 1:1 reference line. The printed
**coverage** is the fraction of actual values that fall inside the P10–P90
band — a well-calibrated 80% interval targets ~0.80.

In [ ]:
dval = xgb.DMatrix(X_val, feature_names=list(FEATURE_NAMES))
preds = booster.predict(dval)   # columns = P10, P50, P90
p10v, p50v, p90v = preds[:, 0], preds[:, 1], preds[:, 2]

err = p50v - y_val
mae = float(np.mean(np.abs(err)))
rmse = float(np.sqrt(np.mean(err ** 2)))
r2 = float(1.0 - np.sum(err ** 2) / np.sum((y_val - y_val.mean()) ** 2))
coverage = float(np.mean((y_val >= p10v) & (y_val <= p90v)))
print(f"MAE  {mae:.4f} arcsec")
print(f"RMSE {rmse:.4f} arcsec")
print(f"R^2  {r2:.4f}")
print(f"P10-P90 coverage: {coverage:.3f} (ideal ~0.80)")

lo_err = np.clip(p50v - p10v, 0, None)
hi_err = np.clip(p90v - p50v, 0, None)
fig, ax = plt.subplots(figsize=(6, 6))
ax.errorbar(
    y_val, p50v, yerr=[lo_err, hi_err], fmt="o", ms=3, alpha=0.35,
    ecolor="lightgray", color="steelblue", label="P50 ± [P10, P90]",
)
lims = [min(y_val.min(), p50v.min()), max(y_val.max(), p50v.max())]
ax.plot(lims, lims, "r--", lw=1, label="1:1")
ax.set_xlabel("actual seeing (arcsec)")
ax.set_ylabel("predicted P50 (arcsec)")
ax.set_title(f"Predicted vs actual (val)  —  R²={r2:.2f}, coverage={coverage:.2f}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sample night forecast

ERA5 profiles are sampled twice nightly (06:00 & 12:00 UTC), so we take a
contiguous 16-profile window from the chronological validation set as a
representative forecast sequence. The P50 prediction is drawn as a line, the
P10–P90 interval as a shaded band, and the derived ERA5 "actual" seeing is
overlaid as scatter points.

In [ ]:
n = 16
sl = slice(0, n)
idx = np.arange(n)
p10w, p50w, p90w = p10v[sl], p50v[sl], p90v[sl]
actual = y_val[sl]

in_band = int(np.sum((actual >= p10w) & (actual <= p90w)))
print(f"window coverage: {in_band}/{n} actuals inside P10-P90 band")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(idx, p50w, "-", color="navy", lw=2, label="P50 forecast")
ax.fill_between(idx, p10w, p90w, color="navy", alpha=0.18, label="P10–P90 band")
ax.scatter(idx, actual, color="darkorange", zorder=5, label="actual (ERA5)")
ax.set_xlabel("validation profile (chronological, ~2 per night)")
ax.set_ylabel("seeing FWHM (arcsec)")
ax.set_title("Sample forecast sequence: predicted band vs actual")
ax.legend()
plt.tight_layout()
plt.show()

## 6. ERA5 upgrade path

This model is trained on a single site (Stanford, CA). `api/ml/harvest_stanford.py`
is a thin driver over `api/ml/era5.py` and is the template for extending the
training set to **more locations and more years**:

1. **More years.** Edit `START_DATE` / `END_DATE` in `harvest_stanford.py`.
   Surface fields are pulled per-year from the ARCO single-levels *timeseries*
   dataset (one request per year); pressure-level profiles are pulled in
   quarterly synoptic chunks (`MONTHS_PER_CHUNK`) sampled at the nighttime
   `PRESSURE_HOURS_UTC` to stay under the CDS per-request cost cap. The harvest
   is resumable — existing `.nc` files are skipped.

2. **More locations.** Copy `harvest_stanford.py`, change `LAT` / `LON` and the
   output base path (`NC_BASE` / `NPZ_OUT`), and re-run. `PRESSURE_HOURS_UTC`
   should track the site's local deep-night hours. Build each site into its own
   `.npz`, then concatenate the `X` / `y` arrays before training to learn a
   multi-site model. Adding a one-hot or lat/lon site feature lets the model
   condition on location.

3. **Retrain & re-evaluate.** Run
   `python -m api.ml.check_distribution` to confirm the labels stay log-normal,
   then
   `python -m api.ml.train_xgb --source era5 --era5-cache api/ml/data/<site>.npz`
   and re-run this notebook to compare feature importance and calibration
   across sites.

4. **Toward ground truth.** The current label is physics-derived, not measured.
   Joining to on-site DIMM seeing monitor archives as the regression target
   would calibrate absolute values; the 27-feature contract and the rest of the
   pipeline stay unchanged.